# 01 — Cleaning, Sentiment & Features

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import html
import re
import unicodedata

SHOP_COLUMNS = ["Shop_ID", "Shop_Name", "Shop_type", "Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher", "Target_Label", "Time_Collected"]
PRODUCT_COLUMNS = ["Shop_ID", "product_id", "product_name", "product_details", "description_text", "Review_stars"]
REVIEW_COLUMNS = ["Shop_ID", "product_id", "review_id", "user_name", "rating", "review_time", "review_text", "has_image"]

def load_json_folder(folder, pattern, columns):
    records = []
    for path in sorted(folder.glob(pattern)):
        with path.open(encoding="utf-8") as handle:
            records.extend(json.load(handle))
    return pd.DataFrame(records) if records else pd.DataFrame(columns=columns)

shops = pd.read_excel(project_path("shops"), dtype={"Shop_ID": str})
products = load_json_folder(project_path("products"), "shop_*_products.json", PRODUCT_COLUMNS)
reviews = load_json_folder(project_path("reviews"), "shop_*_reviews.json", REVIEW_COLUMNS)

In [ ]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", html.unescape(str(value))).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"([!?.,])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"\s+([!?.,])", r"\1", text)

products_clean = products.copy()
reviews_clean = reviews.copy()
products_clean["description_clean"] = products_clean["description_text"].map(clean_text)
reviews_clean["review_clean"] = reviews_clean["review_text"].map(clean_text)
reviews_clean["Comment_Length"] = reviews_clean["review_text"].fillna("").astype(str).str.len()
reviews_clean["Word_Count"] = reviews_clean["review_text"].fillna("").astype(str).str.split().str.len()

In [ ]:
POSITIVE = {"tốt", "đẹp", "ưng", "ok", "ổn", "nhanh", "chuẩn", "tuyệt", "hài lòng", "chất lượng"}
NEGATIVE = {"tệ", "xấu", "chậm", "lỗi", "hỏng", "kém", "thất vọng", "giả", "không tốt", "không hài lòng"}

def sentiment_score(text):
    value = "" if text is None or pd.isna(text) else str(text).lower()
    tokens = set(re.findall(r"\w+", value, flags=re.UNICODE))
    positive = sum(term in value if " " in term else term in tokens for term in POSITIVE)
    negative = sum(term in value if " " in term else term in tokens for term in NEGATIVE)
    return float((positive - negative) / max(positive + negative, 1))

reviews_clean["Sentiment_Score"] = reviews_clean["review_clean"].map(sentiment_score)
reviews_clean["Sentiment_Label"] = pd.cut(reviews_clean["Sentiment_Score"], [-1.01, -0.01, 0.01, 1.01], labels=["negative", "neutral", "positive"])
display(reviews_clean.head())

In [ ]:
shops.to_csv(PROCESSED / "shops_clean.csv", index=False)
products_clean.to_csv(PROCESSED / "products_clean.csv", index=False)
reviews_clean.to_csv(PROCESSED / "reviews_clean.csv", index=False)
merged = reviews_clean.merge(products_clean, on=["Shop_ID", "product_id"], how="left", validate="many_to_one", suffixes=("_review", "_product")).merge(shops, on="Shop_ID", how="left", validate="many_to_one")
merged.to_csv(PROCESSED / "merged_dataset.csv", index=False)
print("Saved processed datasets:", {"shops": len(shops), "products": len(products_clean), "reviews": len(reviews_clean), "merged": len(merged)})